In [3]:
!pip install openai==0.28

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 1.6 MB/s eta 0:00:00


In [4]:
import os
import openai

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

openai.api_key = os.getenv("OPENAI_API_KEY")

In [5]:
import json

def get_current_weather(location, unit = 'celsius'):

  weather_info = {
      "location": location,
      "temperature": "24",
      "unit": unit,
      "forecast" : ["sunny","windy"]
  }

  return json.dumps(weather_info)

In [6]:
get_current_weather("서울")

'{"location": "\\uc11c\\uc6b8", "temperature": "24", "unit": "celsius", "forecast": ["sunny", "windy"]}'

In [17]:
get_current_weather("seoul")

'{"location": "seoul", "temperature": "24", "unit": "celsius", "forecast": ["sunny", "windy"]}'

In [7]:
def get_gps():
  return '서울'

In [33]:
# 함수 호출 위한 dict

available_functions = {
    'get_current_weather': get_current_weather,
    'get_location_from_gps' : get_gps
}

In [34]:
available_functions['get_current_weather']('서울')

'{"location": "\\uc11c\\uc6b8", "temperature": "24", "unit": "celsius", "forecast": ["sunny", "windy"]}'

In [35]:
available_functions['get_current_weather']('seoul')

'{"location": "seoul", "temperature": "24", "unit": "celsius", "forecast": ["sunny", "windy"]}'

In [37]:
available_functions['get_location_from_gps']()

'서울'

In [38]:
def run_convesation(input_text):
  # 1. converstaion + available_functions >> gpt
  messages=[
      {"role": "user", "content": input_text}
  ]

  functions = [
      {
          "name": "get_current_weather",
          "description":"Get the current weather in a given location",
          "parameters": {
              "type":'object',
              "properties": {
                  "location":{
                      "type": "string",
                      "description": "The city and state, e.g. San Francisco, CA"
                  },
                  "unit":{
                      "type": "string",
                      "enum": ["celsius", "fahrenheit"]
                  },
                  },
              "required":['location'],
          },
      }
  ]

  response = openai.ChatCompletion.create(
      model="gpt-3.5-turbo-0613",
      messages=messages,
      functions=functions,
      function_call="auto",
  )
  print(response)

  response_messages = response['choices'][0]['message']
  print(response_messages)

  # function_call : 함수 특정
  if response_messages.get('function_call'):

    # 함수의 이름과 인자를 가져온다
    function_name = response_messages['function_call']['name']
    function_args = json.loads(response_messages['function_call']['arguments'])

    # response_message : function_call, name, arguments
    messages.append(response_messages)

    # 함수 호출해서 결과 값 얻기
    function_response = available_functions[function_name](
        location = function_args.get('location'),
        unit = function_args.get('unit')
    )
    # 함수 호출 결과 >> 메시지에 추가
    messages.append({
        "role": "function",
        "name": function_name,
        "content": function_response
    })
    # 역할 : 함수 호출결과는 역할을 function 으로 설정
    print(messages)

    second_response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo-0613",
        messages=messages,
        # 사용자 요청사항, 함수 이름 및 인자, 함수 호출 결과
    )
    second_response = second_response['choices'][0]['message']

    return second_response
  else:
    return response

In [39]:
run_convesation("오늘 날씨는 어때?")

{
  "id": "chatcmpl-9bRxMHjiW2cyM109vqHhbF9uA8nqq",
  "object": "chat.completion",
  "created": 1718712628,
  "model": "gpt-3.5-turbo-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "function_call": {
          "name": "get_current_weather",
          "arguments": "{\n  \"location\": \"Seoul\"\n}"
        }
      },
      "logprobs": null,
      "finish_reason": "function_call"
    }
  ],
  "usage": {
    "prompt_tokens": 88,
    "completion_tokens": 17,
    "total_tokens": 105
  },
  "system_fingerprint": null
}
{
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_current_weather",
    "arguments": "{\n  \"location\": \"Seoul\"\n}"
  }
}
[{'role': 'user', 'content': '오늘 날씨는 어때?'}, <OpenAIObject at 0x7b59b9fff290> JSON: {
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_current_weather",
    "arguments": "{\n  \"location\": \"Seoul\"\n}"
  }
}, {'ro

<OpenAIObject at 0x7b59b9ffd620> JSON: {
  "role": "assistant",
  "content": "\uc624\ub298 \uc11c\uc6b8\uc758 \ub0a0\uc528\ub294 \ub9d1\uace0 \ubc14\ub78c\uc774 \uc870\uae08 \uac15\ud558\ub124\uc694. \uae30\uc628\uc740 24\ub3c4\uc785\ub2c8\ub2e4."
}

In [40]:
run_convesation("What is the weather in Seoul?")

{
  "id": "chatcmpl-9bRxPDZ2sxHYrblungLubEVGOBJjY",
  "object": "chat.completion",
  "created": 1718712631,
  "model": "gpt-3.5-turbo-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "function_call": {
          "name": "get_current_weather",
          "arguments": "{\n  \"location\": \"Seoul\"\n}"
        }
      },
      "logprobs": null,
      "finish_reason": "function_call"
    }
  ],
  "usage": {
    "prompt_tokens": 81,
    "completion_tokens": 17,
    "total_tokens": 98
  },
  "system_fingerprint": null
}
{
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_current_weather",
    "arguments": "{\n  \"location\": \"Seoul\"\n}"
  }
}
[{'role': 'user', 'content': 'What is the weather in Seoul?'}, <OpenAIObject at 0x7b59b9ffdbc0> JSON: {
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_current_weather",
    "arguments": "{\n  \"location\": \"Seoul

<OpenAIObject at 0x7b59ba069170> JSON: {
  "role": "assistant",
  "content": "The weather in Seoul is currently sunny and windy, with a temperature of 24 degrees."
}

In [41]:
# 코드 수정 (Refactoring)

def run_conversation(user_input_text):
  # 1. converstaion + available_functions >> gpt
  messages=[
      {"role": "user", "content": user_input_text}
  ]

  functions = [
      {
          "name": "get_current_weather",
          "description":"Get the current weather in a given location",
          "parameters": {
              "type":'object',
              "properties": {
                  "location":{
                      "type": "string",
                      "description": "The city and state, e.g. San Francisco, CA"
                  },
                  "unit":{
                      "type": "string",
                      "enum": ["celsius", "fahrenheit"]
                  },
                  },
              "required":['location'],
          },
      },
      {
          "name" : "get_location_from_gps",
          "description" : "Get the current location",
          "parameters":{
              "type": "object",
              "properties":{
                  "unit": {"type":"string"},
              },
              "required": [],
          },
      }
  ]

  response = openai.ChatCompletion.create(
      model="gpt-3.5-turbo-0613",
      messages=messages,
      functions=functions,
      function_call="auto",
  )
  print(response)

  response_messages = response['choices'][0]['message']
  print(response_messages)

  # gpt 가 function call을 원하는 지 확인
  # function_call : 함수 특정
  if response_messages.get('function_call'):

    # 함수의 이름과 인자를 가져온다
    function_name = response_messages['function_call']['name']
    function_args = json.loads(response_messages['function_call']['arguments'])

    # 함수 호출
    # 어떤 함수가 호출되는 지 모르기 때문에, 즉 인자를 특정할 수 없다.
    function_response = available_functions[function_name](**function_args)

    # [중요] 함수 이름과 인자 정보 추가
    messages.append(response_messages)

    # 함수 호출 결과 >> 메시지에 추가
    messages.append({
        "role": "function",
        "name": function_name,
        "content": function_response
    })
    # 역할 : 함수 호출결과는 역할을 function 으로 설정
    print(messages)

    second_response = openai.ChatCompletion.create(
         model="gpt-3.5-turbo-0613",
         messages=messages,
        # 사용자 요청사항, 함수 이름 및 인자, 함수 호출 결과
    )
    second_response = second_response['choices'][0]['message']

    return second_response
  else:
    return response

In [45]:
message = run_conversation("현재 위치는 어디인가요?")
message

{
  "id": "chatcmpl-9bRyd2wwVDkwZ9TzX8n3G7TTdzfPm",
  "object": "chat.completion",
  "created": 1718712707,
  "model": "gpt-3.5-turbo-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "function_call": {
          "name": "get_location_from_gps",
          "arguments": "{}"
        }
      },
      "logprobs": null,
      "finish_reason": "function_call"
    }
  ],
  "usage": {
    "prompt_tokens": 108,
    "completion_tokens": 9,
    "total_tokens": 117
  },
  "system_fingerprint": null
}
{
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_location_from_gps",
    "arguments": "{}"
  }
}
[{'role': 'user', 'content': '현재 위치는 어디인가요?'}, <OpenAIObject at 0x7b59ba069da0> JSON: {
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_location_from_gps",
    "arguments": "{}"
  }
}, {'role': 'function', 'name': 'get_location_from_gps', 'content': '서울'}]


<OpenAIObject at 0x7b59ba06a480> JSON: {
  "role": "assistant",
  "content": "\ud604\uc7ac \uc704\uce58\ub294 \uc11c\uc6b8\uc785\ub2c8\ub2e4."
}

In [44]:
message.content

'제 현재 위치는 서울입니다.'

In [46]:
message = run_conversation("서울의 날씨는요?")
message

{
  "id": "chatcmpl-9bRywzBbFkHN9rYszCuGK9NsCDabc",
  "object": "chat.completion",
  "created": 1718712726,
  "model": "gpt-3.5-turbo-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "function_call": {
          "name": "get_current_weather",
          "arguments": "{\n  \"location\": \"Seoul, South Korea\"\n}"
        }
      },
      "logprobs": null,
      "finish_reason": "function_call"
    }
  ],
  "usage": {
    "prompt_tokens": 109,
    "completion_tokens": 20,
    "total_tokens": 129
  },
  "system_fingerprint": null
}
{
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_current_weather",
    "arguments": "{\n  \"location\": \"Seoul, South Korea\"\n}"
  }
}
[{'role': 'user', 'content': '서울의 날씨는요?'}, <OpenAIObject at 0x7b59ba069da0> JSON: {
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "get_current_weather",
    "arguments": "{\n  \"location\":

<OpenAIObject at 0x7b59ba07c5e0> JSON: {
  "role": "assistant",
  "content": "\uc11c\uc6b8\uc758 \ud604\uc7ac \ub0a0\uc528\ub294 24\u00b0C\ub85c \ub9d1\uace0 \ubc14\ub78c\uc774 \uc870\uae08 \uc788\uc2b5\ub2c8\ub2e4."
}

In [47]:
message.content

'서울의 현재 날씨는 24°C로 맑고 바람이 조금 있습니다.'